# Combinador de Excel — maqueta en pandas

Este notebook es el **banco de pruebas** del combinador. El flujo acordado es:

1. Cargar los Excel **manualmente** (lista de rutas o carpeta `data/entrada`).
2. Inspeccionar la **fila 1** de cada archivo: ahí están los títulos.
3. Elegir un **archivo de referencia**: sus títulos serán el encabezado del consolidado.
4. Mapear **a mano** las columnas de los demás archivos hacia ese encabezado.
5. Decidir qué hacer con las columnas extra.
6. Generar un `.xlsx` consolidado con trazabilidad.

> **Definición acordada:** encabezado = fila 1. La fila 1 de los archivos que no son referencia se descarta; esos archivos solo aportan datos.

**Índice**

1. Preparar el entorno del notebook
2. Carga manual de archivos
3. Inventario: hojas y primeras filas
4. Elegir el archivo de referencia
5. Leer cada archivo y comparar sus columnas
6. Mapeo manual de columnas
7. Columnas extra: avisar y decidir
8. Validación antes de combinar
9. Construir el consolidado
10. Diagnóstico
11. Exportar
12. Función reutilizable (puente hacia la GUI)


## 1. Preparar el entorno del notebook

Importamos las librerías y definimos las rutas. `RAIZ` se calcula a partir del directorio de trabajo: funciona tanto si abrís JupyterLab desde la raíz del proyecto como si el kernel arranca dentro de `notebooks/`.

También definimos qué extensiones aceptamos. `.xlsx` y `.xlsm` los lee `openpyxl`; `.xls` antiguos los lee `xlrd`.


In [1]:
from __future__ import annotations

import re
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# El notebook vive en notebooks/. Si el kernel arranca en la raíz, también funciona.
RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

DIR_ENTRADA = RAIZ / "data" / "entrada"
DIR_SALIDA = RAIZ / "data" / "salida"
EXTENSIONES = {".xlsx", ".xls", ".xlsm"}

DIR_SALIDA.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ)
print("Entrada:", DIR_ENTRADA)
print("Salida:", DIR_SALIDA)
print("pandas:", pd.__version__)


Raíz del proyecto: /home/luis/Desktop/CORPOSALUD-TOOLS
Entrada: /home/luis/Desktop/CORPOSALUD-TOOLS/data/entrada
Salida: /home/luis/Desktop/CORPOSALUD-TOOLS/data/salida
pandas: 3.1.0


## 2. Carga manual de archivos

Hay dos formas de elegir los archivos:

- **Lista explícita:** completá `ARCHIVOS_MANUALES` con las rutas que quieras.
- **Carpeta:** con `USAR_CARPETA = True` se toman todos los Excel que estén en `data/entrada/`.

Si `ARCHIVOS_MANUALES` tiene elementos, tiene prioridad sobre la carpeta. Los archivos temporales de Excel (`~$...`) se ignoran automáticamente.

Para esta primera prueba dejé tres archivos demo en `data/entrada/`.


In [ ]:
ARCHIVOS_MANUALES: list[str] = [
    # "data/entrada/mi_archivo.xlsx",
]
USAR_CARPETA = True


def listar_excels(carpeta: Path) -> list[Path]:
    if not carpeta.exists():
        return []
    return sorted(
        p
        for p in carpeta.iterdir()
        if p.is_file()
        and p.suffix.lower() in EXTENSIONES
        and not p.name.startswith("~$")
    )


if ARCHIVOS_MANUALES:
    ARCHIVOS = [Path(p).expanduser().resolve() for p in ARCHIVOS_MANUALES]
else:
    ARCHIVOS = listar_excels(DIR_ENTRADA)

if not ARCHIVOS:
    raise FileNotFoundError(
        f"No hay archivos Excel en {DIR_ENTRADA}. "
        "Copiá tus archivos ahí o completá ARCHIVOS_MANUALES."
    )

print(f"{len(ARCHIVOS)} archivo(s) para procesar:\n")
for i, ruta in enumerate(ARCHIVOS, 1):
    print(f"{i}. {ruta.name}  ({ruta.stat().st_size / 1024:.1f} KB)")


3 archivo(s) para procesar:

1. demo_pacientes_enero.xlsx  (4.9 KB)
2. demo_pacientes_febrero.xlsx  (4.9 KB)
3. demo_pacientes_marzo.xlsx  (5.6 KB)


## 3. Inventario: hojas y primeras filas

Antes de transformar nada, miramos cada libro **en crudo**. Leemos con `header=None` para que pandas no interprete la fila 1: así vemos exactamente qué hay en cada celda.

Esto sirve para detectar problemas temprano:

- hojas con nombres inesperados,
- títulos con saltos de línea o espacios,
- filas de título/logos arriba del encabezado,
- columnas vacías al final.


In [ ]:
def hojas_del_libro(ruta: Path) -> list[str]:
    with pd.ExcelFile(ruta) as libro:
        return libro.sheet_names


def leer_crudo(ruta: Path, hoja: str | int = 0) -> pd.DataFrame:
    """Lee sin interpretar encabezados: la fila 0 es la fila 1 de Excel."""
    return pd.read_excel(
        ruta,
        sheet_name=hoja,
        header=None,
        dtype=str,
        keep_default_na=False,
    )


for ruta in ARCHIVOS:
    print("=" * 90)
    print(f"ARCHIVO: {ruta.name}")
    try:
        hojas = hojas_del_libro(ruta)
        print(f"Hojas: {hojas}")
        primera_hoja = hojas[0]
        crudo = leer_crudo(ruta, primera_hoja)
        print(f"Hoja inspeccionada: {primera_hoja!r} \n Dimensiones: {crudo.shape[0]} filas x {crudo.shape[1]} columnas")
        display(crudo.head(5))
    except Exception as exc:
        print(f"⚠️ No se pudo leer: {exc}")


ARCHIVO: demo_pacientes_enero.xlsx
Hojas: ['DATOS']
Hoja inspeccionada: 'DATOS'
Dimensiones: 4 filas x 5 columnas


,0,1,2,3,4
0,CEDULA,NOMBRE,FECHA,SEDE,EDAD
1,01234567,Ana Pérez,2026-01-05,Caracas,34
2,12345678,Luis Gómez,2026-01-07,Valencia,41
3,23456789,María Rodríguez,2026-01-12,Maracaibo,29


ARCHIVO: demo_pacientes_febrero.xlsx
Hojas: ['DATOS']
Hoja inspeccionada: 'DATOS'
Dimensiones: 3 filas x 6 columnas


,0,1,2,3,4,5
0,C.I.,NOMBRE COMPLETO,FECHA_ATENCION,SEDE,EDAD,CORREO
1,01234567,Ana Pérez,2026-02-03,Caracas,34,ana@example.com
2,34567890,Pedro Díaz,2026-02-08,Barquisimeto,52,pedro@example.com


ARCHIVO: demo_pacientes_marzo.xlsx
Hojas: ['DATOS', 'HOJA_ALTERNATIVA']
Hoja inspeccionada: 'DATOS'
Dimensiones: 3 filas x 6 columnas


,0,1,2,3,4,5
0,SEDE,CEDULA,EDAD,NOMBRES Y APELLIDOS,FECHA,OBSERVACION
1,Maracay,45678901,38,José Martínez,2026-03-02,control
2,Caracas,01234567,34,Ana Pérez,2026-03-09,seguimiento


## 4. Elegir el archivo de referencia

El encabezado del consolidado sale de la **fila 1 del archivo que elijas**. Los demás archivos no aportan títulos: su fila 1 se descarta y solo se usan sus datos.

Reglas que aplicamos sobre el archivo de referencia:

- La fila 1 debe tener al menos un título.
- Se recortan las columnas vacías del final.
- No se permiten títulos vacíos en el medio ni títulos duplicados, porque harían ambiguo el mapeo.

Cambiá `ARCHIVO_REFERENCIA` por el nombre de uno de los archivos del paso 2. Si el archivo demo no está, el notebook elige el primero de la lista.


In [4]:
ARCHIVO_REFERENCIA = "demo_pacientes_enero.xlsx"
HOJA_REFERENCIA: str | int = 0

# Si algún archivo necesita una hoja distinta a la primera, indicalo acá.
HOJAS_POR_ARCHIVO: dict[str, str | int] = {
    # "demo_pacientes_marzo.xlsx": "DATOS",
}


def limpiar_titulo(valor: object) -> str:
    if pd.isna(valor): return ""
    texto = str(valor).replace("\n", " ")
    return re.sub(r"\s+", " ", texto).strip()


def recortar_titulos(titulos: list[str]) -> list[str]:
    while titulos and titulos[-1] == "":
        titulos.pop()
    return titulos


def validar_titulos(titulos: list[str], etiqueta: str) -> None:
    vacios = [i + 1 for i, titulo in enumerate(titulos) if titulo == ""]
    if vacios:
        raise ValueError(
            f"{etiqueta}: hay títulos vacíos en las posiciones {vacios}. "
            "Completalos en el Excel antes de combinar."
        )
    repetidos = sorted({t for t in titulos if titulos.count(t) > 1})
    if repetidos:
        raise ValueError(
            f"{etiqueta}: títulos duplicados {repetidos}. "
            "Cada columna necesita un nombre único."
        )


def buscar_archivo(nombre: str) -> Path:
    coincidencias = [p for p in ARCHIVOS if p.name == nombre]
    if not coincidencias:
        disponibles = ", ".join(p.name for p in ARCHIVOS)
        raise FileNotFoundError(f"No encontré {nombre!r}. Disponibles: {disponibles}")
    return coincidencias[0]


# Si el archivo demo no existe, usamos el primero disponible.
if not any(p.name == ARCHIVO_REFERENCIA for p in ARCHIVOS):
    ARCHIVO_REFERENCIA = ARCHIVOS[0].name
    print(f"⚠️ Usando como referencia el primer archivo: {ARCHIVO_REFERENCIA}")

RUTA_REFERENCIA = buscar_archivo(ARCHIVO_REFERENCIA)
crudo_ref = leer_crudo(RUTA_REFERENCIA, HOJA_REFERENCIA)
if crudo_ref.empty:
    raise ValueError(f"{RUTA_REFERENCIA.name} está vacío.")

ENCABEZADO_REF = recortar_titulos(
    [limpiar_titulo(valor) for valor in crudo_ref.iloc[0].tolist()]
)
validar_titulos(ENCABEZADO_REF, f"Referencia {RUTA_REFERENCIA.name}")

print(f"Archivo de referencia: {RUTA_REFERENCIA.name}")
print(f"Hoja: {HOJA_REFERENCIA!r}")
print(f"Columnas del encabezado ({len(ENCABEZADO_REF)}):\n")
display(
    pd.DataFrame(
        {"posición": range(1, len(ENCABEZADO_REF) + 1), "título": ENCABEZADO_REF}
    )
)


Archivo de referencia: demo_pacientes_enero.xlsx
Hoja: 0
Columnas del encabezado (5):



,posición,título
0,1,CEDULA
1,2,NOMBRE
2,3,FECHA
3,4,SEDE
4,5,EDAD


## 5. Leer cada archivo y comparar sus columnas

Acá convertimos la fila 1 en nombres de columna y separamos los datos (fila 2 en adelante). Para cada archivo calculamos:

- **coinciden:** columnas con el mismo título que la referencia.
- **faltan:** la referencia las tiene, el archivo no.
- **sobran:** el archivo las tiene, la referencia no.

También quitamos filas totalmente vacías, pero conservamos el índice original para poder reportar la **fila real de Excel** en la trazabilidad.

Para archivos que no son referencia, esta comparación es solo informativa: el mapeo real se hace a mano en el paso 6.


In [5]:
def es_vacio(valor: object) -> bool:
    return pd.isna(valor) or str(valor).strip() == ""


def cargar_perfil(ruta: Path, hoja: str | int = 0) -> dict:
    crudo = leer_crudo(ruta, hoja)
    if crudo.empty:
        raise ValueError(f"{ruta.name}: la hoja {hoja!r} está vacía.")

    columnas = recortar_titulos(
        [limpiar_titulo(valor) for valor in crudo.iloc[0].tolist()]
    )
    validar_titulos(columnas, f"{ruta.name} / hoja {hoja!r}")

    datos = crudo.iloc[1:, : len(columnas)].copy()
    datos.columns = columnas

    # Quitamos filas totalmente vacías, conservando el índice original
    # (índice 1 = fila 2 de Excel).
    fila_vacia = datos.apply(
        lambda fila: all(es_vacio(valor) for valor in fila), axis=1
    )
    datos = datos.loc[~fila_vacia]

    return {
        "ruta": ruta,
        "nombre": ruta.name,
        "hoja": hoja,
        "columnas": columnas,
        "datos": datos,
    }


PERFILES = {}
for ruta in ARCHIVOS:
    hoja = HOJAS_POR_ARCHIVO.get(ruta.name, 0)
    try:
        PERFILES[ruta.name] = cargar_perfil(ruta, hoja)
    except Exception as exc:
        print(f"⚠️ {ruta.name}: {exc}")

if RUTA_REFERENCIA.name not in PERFILES:
    raise ValueError("El archivo de referencia no se pudo cargar.")

filas_resumen = []
for nombre, perfil in PERFILES.items():
    columnas = perfil["columnas"]
    filas_resumen.append(
        {
            "archivo": nombre,
            "hoja": str(perfil["hoja"]),
            "filas_datos": len(perfil["datos"]),
            "columnas": len(columnas),
            "coinciden": sum(c in ENCABEZADO_REF for c in columnas),
            "faltan": len([c for c in ENCABEZADO_REF if c not in columnas]),
            "sobran": len([c for c in columnas if c not in ENCABEZADO_REF]),
        }
    )

display(pd.DataFrame(filas_resumen))

for nombre, perfil in PERFILES.items():
    if nombre == RUTA_REFERENCIA.name:
        continue
    faltan = [c for c in ENCABEZADO_REF if c not in perfil["columnas"]]
    sobran = [c for c in perfil["columnas"] if c not in ENCABEZADO_REF]
    print(f"\n{nombre}")
    print(f"  faltan en el archivo: {faltan or '—'}")
    print(f"  sobran en el archivo: {sobran or '—'}")


,archivo,hoja,filas_datos,columnas,coinciden,faltan,sobran
0,demo_pacientes_enero.xlsx,0,3,5,5,0,0
1,demo_pacientes_febrero.xlsx,0,2,6,2,3,4
2,demo_pacientes_marzo.xlsx,0,2,6,4,1,2



demo_pacientes_febrero.xlsx
  faltan en el archivo: ['CEDULA', 'NOMBRE', 'FECHA']
  sobran en el archivo: ['C.I.', 'NOMBRE COMPLETO', 'FECHA_ATENCION', 'CORREO']

demo_pacientes_marzo.xlsx
  faltan en el archivo: ['NOMBRE']
  sobran en el archivo: ['NOMBRES Y APELLIDOS', 'OBSERVACION']


## 6. Mapeo manual de columnas

Para cada archivo armamos un diccionario con esta forma:

```python
{"COLUMNA_REFERENCIA": "COLUMNA_EN_ESTE_ARCHIVO"}
```

Reglas:

- Si el valor es `None`, esa columna queda **vacía** para ese archivo.
- El notebook **sugiere** coincidencias comparando nombres normalizados (mayúsculas, sin acentos, sin puntuación). Es solo un punto de partida.
- Vos corregís lo que haga falta en `AJUSTES_MANUALES`.
- Con `USAR_SUGERENCIAS = False` arrancás todo en `None` y mapeás 100 % a mano.
- La columna de referencia y la columna fuente deben existir; si no, la validación del paso 8 lo marca.

En los demo, las sugerencias aciertan `CEDULA`, `SEDE`, `EDAD` y `FECHA` cuando el nombre es idéntico, pero **no** pueden adivinar `C.I.`, `NOMBRE COMPLETO` ni `FECHA_ATENCION`. Por eso están cargados en `AJUSTES_MANUALES`.


In [6]:
def normalizar_titulo(texto: str) -> str:
    sin_acentos = "".join(
        c
        for c in unicodedata.normalize("NFKD", str(texto))
        if not unicodedata.combining(c)
    )
    limpio = re.sub(r"[^A-Za-z0-9]+", " ", sin_acentos.upper())
    return re.sub(r"\s+", " ", limpio).strip()


def sugerir_mapeo(columnas_ref: list[str], columnas_fuente: list[str]) -> dict:
    """Sugiere una columna fuente para cada columna de referencia.

    Es uno-a-uno: una columna fuente no se reutiliza para dos campos.
    """
    usadas: set[str] = set()
    sugerencias = {}
    for col_ref in columnas_ref:
        candidatas = [c for c in columnas_fuente if c not in usadas]
        if not candidatas:
            sugerencias[col_ref] = None
            continue
        normal_ref = normalizar_titulo(col_ref)
        puntajes = [
            (c, SequenceMatcher(None, normal_ref, normalizar_titulo(c)).ratio())
            for c in candidatas
        ]
        mejor, puntaje = max(puntajes, key=lambda par: par[1])
        if puntaje >= 0.85:
            sugerencias[col_ref] = mejor
            usadas.add(mejor)
        else:
            sugerencias[col_ref] = None
    return sugerencias


USAR_SUGERENCIAS = True

# Acá corregís lo que la sugerencia no pueda adivinar.
# Formato: {archivo: {columna_referencia: columna_del_archivo_o_None}}
AJUSTES_MANUALES: dict[str, dict[str, str | None]] = {
    "demo_pacientes_febrero.xlsx": {
        "CEDULA": "C.I.",
        "NOMBRE": "NOMBRE COMPLETO",
        "FECHA": "FECHA_ATENCION",
    },
    "demo_pacientes_marzo.xlsx": {
        "NOMBRE": "NOMBRES Y APELLIDOS",
    },
}


def construir_mapeo(
    perfiles: dict,
    columnas_ref: list[str],
    ajustes: dict[str, dict[str, str | None]],
    usar_sugerencias: bool = True,
) -> dict[str, dict[str, str | None]]:
    mapeo = {}
    for nombre, perfil in perfiles.items():
        if usar_sugerencias:
            mapa = sugerir_mapeo(columnas_ref, perfil["columnas"])
        else:
            mapa = {col: None for col in columnas_ref}

        for col_ref, col_fuente in ajustes.get(nombre, {}).items():
            if col_ref not in columnas_ref:
                raise KeyError(
                    f"{nombre}: {col_ref!r} no existe en el encabezado de referencia."
                )
            mapa[col_ref] = col_fuente
        mapeo[nombre] = mapa
    return mapeo


MAPEO = construir_mapeo(PERFILES, ENCABEZADO_REF, AJUSTES_MANUALES, USAR_SUGERENCIAS)

tabla_mapeo = pd.DataFrame(
    {
        nombre: [mapa.get(col) or "—" for col in ENCABEZADO_REF]
        for nombre, mapa in MAPEO.items()
    },
    index=ENCABEZADO_REF,
)
tabla_mapeo.index.name = "columna de referencia"
display(tabla_mapeo)


,demo_pacientes_enero.xlsx,demo_pacientes_febrero.xlsx,demo_pacientes_marzo.xlsx
columna de referencia,,,
CEDULA,CEDULA,C.I.,CEDULA
NOMBRE,NOMBRE,NOMBRE COMPLETO,NOMBRES Y APELLIDOS
FECHA,FECHA,FECHA_ATENCION,FECHA
SEDE,SEDE,SEDE,SEDE
EDAD,EDAD,EDAD,EDAD


## 7. Columnas extra: avisar y decidir

Una columna extra es la que existe en un archivo, **no** está en el encabezado de referencia y **tampoco se usa como fuente de un campo mapeado**. No se incluye nunca por accidente: primero se listan todas y después vos decidís.

Las columnas que sí se usaron para mapear se muestran aparte, para que no se confundan con las sobrantes.

- `INCLUIR_TODOS_LOS_EXTRAS = False`: solo se incluyen las que estén en `EXTRAS_A_INCLUIR`.
- Si ponés `True`, se incluyen todas las detectadas.
- Si una extra se llama igual que una columna de referencia, se rechaza (ya existe).
- Si una extra es la fuente de un campo mapeado, también se rechaza para no duplicar datos.

En el demo, `CORREO` (febrero) se conserva y `OBSERVACION` (marzo) se descarta a propósito, para que veas ambos casos.


In [7]:
EXTRAS_DETECTADAS: dict[str, list[str]] = {}

print("Columnas fuera del encabezado de referencia:\n")
for nombre, perfil in PERFILES.items():
    if nombre == RUTA_REFERENCIA.name:
        continue
    mapeadas = {v for v in MAPEO[nombre].values() if v}
    fuera = [c for c in perfil["columnas"] if c not in ENCABEZADO_REF]
    usadas = [c for c in fuera if c in mapeadas]
    extras = [c for c in fuera if c not in mapeadas]
    if extras:
        EXTRAS_DETECTADAS[nombre] = extras
    print(f"- {nombre}")
    print(f"    usadas para mapear: {usadas or '—'}")
    print(f"    extras reales:      {extras or '—'}")

if not EXTRAS_DETECTADAS:
    print("\nNo hay columnas extra que decidir.")

INCLUIR_TODOS_LOS_EXTRAS = False

EXTRAS_A_INCLUIR: dict[str, list[str]] = {
    "demo_pacientes_febrero.xlsx": ["CORREO"],
    # "demo_pacientes_marzo.xlsx": ["OBSERVACION"],
}

# Ignoramos configuración de archivos que no se cargaron (por ejemplo, si
# reemplazaste los demo por datos reales).
for nombre in list(EXTRAS_A_INCLUIR):
    if nombre not in PERFILES:
        print(f"⚠️ {nombre!r} no está cargado; se ignora su configuración de extras.")
        EXTRAS_A_INCLUIR.pop(nombre)

if INCLUIR_TODOS_LOS_EXTRAS:
    EXTRAS_A_INCLUIR = {n: list(cols) for n, cols in EXTRAS_DETECTADAS.items()}

print("\nExtras que se van a conservar:", EXTRAS_A_INCLUIR or "ninguna")


Columnas fuera del encabezado de referencia:

- demo_pacientes_febrero.xlsx
    usadas para mapear: ['C.I.', 'NOMBRE COMPLETO', 'FECHA_ATENCION']
    extras reales:      ['CORREO']
- demo_pacientes_marzo.xlsx
    usadas para mapear: ['NOMBRES Y APELLIDOS']
    extras reales:      ['OBSERVACION']

Extras que se van a conservar: {'demo_pacientes_febrero.xlsx': ['CORREO']}


## 8. Validación antes de combinar

Antes de construir nada, revisamos que la configuración sea coherente:

- que cada columna de referencia exista en el archivo de referencia,
- que cada columna fuente exista en su archivo,
- que no se use la misma columna fuente para dos campos,
- que las extras no choquen con la referencia ni con el mapeo,
- que no haya extras llamadas `archivo_origen`, `hoja_origen` o `fila_origen` (esas las reserva el consolidado).

Si `EXIGIR_MAPEO_COMPLETO = True`, cualquier columna sin mapear se trata como error. Por defecto es `False`: una columna sin mapear queda vacía y se avisa.

También mostramos qué columnas fuente se van a **descartar** (las que no se mapearon ni se incluyeron como extra).


In [8]:
EXIGIR_MAPEO_COMPLETO = False
RESERVADAS = {"archivo_origen", "hoja_origen", "fila_origen"}


def validar_configuracion(
    perfiles: dict,
    mapeo: dict[str, dict[str, str | None]],
    extras: dict[str, list[str]],
    columnas_ref: list[str],
    exigir_mapeo_completo: bool = False,
) -> list[str]:
    problemas = []
    for nombre, mapa in mapeo.items():
        perfil = perfiles[nombre]
        usadas: list[str] = []

        for col_ref, col_fuente in mapa.items():
            if col_ref not in columnas_ref:
                problemas.append(
                    f"{nombre}: {col_ref!r} no está en el encabezado de referencia."
                )
            if col_fuente is None:
                continue
            if col_fuente not in perfil["columnas"]:
                problemas.append(
                    f"{nombre}: {col_ref!r} apunta a {col_fuente!r}, "
                    "que no existe en ese archivo."
                )
                continue
            usadas.append(col_fuente)

        duplicadas = sorted({c for c in usadas if usadas.count(c) > 1})
        if duplicadas:
            problemas.append(
                f"{nombre}: la(s) columna(s) {duplicadas} se usan para más de un campo."
            )

        sin_mapear = [c for c, fuente in mapa.items() if fuente is None]
        if sin_mapear:
            if exigir_mapeo_completo:
                problemas.append(f"{nombre}: columnas sin mapear: {sin_mapear}")
            else:
                print(f"{nombre}: aviso — quedarán vacías: {sin_mapear}")

        for extra in extras.get(nombre, []):
            if extra not in perfil["columnas"]:
                problemas.append(f"{nombre}: la extra {extra!r} no existe.")
            if extra in columnas_ref:
                problemas.append(
                    f"{nombre}: {extra!r} ya es una columna de referencia."
                )
            if extra in usadas:
                problemas.append(
                    f"{nombre}: {extra!r} está incluida como extra y también "
                    "como fuente de un campo."
                )
            if extra in RESERVADAS:
                problemas.append(
                    f"{nombre}: {extra!r} es un nombre reservado por la trazabilidad."
                )
    return problemas


problemas = validar_configuracion(
    PERFILES, MAPEO, EXTRAS_A_INCLUIR, ENCABEZADO_REF, EXIGIR_MAPEO_COMPLETO
)
if problemas:
    raise ValueError("Revisá el mapeo:\n- " + "\n- ".join(problemas))

print("✅ Mapeo válido.\n")
print("Columnas fuente que se descartan:")
alguna_descartada = False
for nombre, perfil in PERFILES.items():
    usadas = {v for v in MAPEO[nombre].values() if v}
    extras = set(EXTRAS_A_INCLUIR.get(nombre, []))
    descartadas = [c for c in perfil["columnas"] if c not in usadas and c not in extras]
    if descartadas:
        alguna_descartada = True
        print(f"- {nombre}: {descartadas}")
if not alguna_descartada:
    print("- ninguna")


✅ Mapeo válido.

Columnas fuente que se descartan:
- demo_pacientes_marzo.xlsx: ['OBSERVACION']


## 9. Construir el consolidado

Por cada archivo:

1. Creamos un DataFrame con **exactamente** las columnas de referencia, en ese orden.
2. Copiamos los datos de la columna mapeada. Si el mapeo es `None`, dejamos vacío.
3. Agregamos las extras aprobadas.
4. Agregamos trazabilidad: `archivo_origen`, `hoja_origen` y `fila_origen` (número de fila real en Excel).
5. Concatenamos todos los archivos.

El resultado tiene el encabezado de la referencia y una fila por cada registro de cada archivo.


In [9]:
def construir_consolidado(
    perfiles: dict,
    mapeo: dict[str, dict[str, str | None]],
    extras: dict[str, list[str]],
    columnas_ref: list[str],
) -> pd.DataFrame:
    partes = []
    for nombre, perfil in perfiles.items():
        datos = perfil["datos"]
        salida = pd.DataFrame(index=datos.index)

        for col_ref in columnas_ref:
            col_fuente = mapeo[nombre].get(col_ref)
            if col_fuente is None:
                salida[col_ref] = ""
            else:
                salida[col_ref] = datos[col_fuente]

        for extra in extras.get(nombre, []):
            salida[extra] = datos[extra]

        salida["archivo_origen"] = nombre
        salida["hoja_origen"] = str(perfil["hoja"])
        salida["fila_origen"] = datos.index + 1  # índice 1 = fila 2 de Excel
        partes.append(salida)

    consolidado = pd.concat(partes, ignore_index=True).fillna("")
    return consolidado


CONSOLIDADO = construir_consolidado(PERFILES, MAPEO, EXTRAS_A_INCLUIR, ENCABEZADO_REF)

print(f"Consolidado: {CONSOLIDADO.shape[0]} filas x {CONSOLIDADO.shape[1]} columnas")
display(CONSOLIDADO.head(10))


Consolidado: 7 filas x 9 columnas


,CEDULA,NOMBRE,FECHA,SEDE,EDAD,archivo_origen,hoja_origen,fila_origen,CORREO
0,01234567,Ana Pérez,2026-01-05,Caracas,34,demo_pacientes_enero.xlsx,0,2,
1,12345678,Luis Gómez,2026-01-07,Valencia,41,demo_pacientes_enero.xlsx,0,3,
2,23456789,María Rodríguez,2026-01-12,Maracaibo,29,demo_pacientes_enero.xlsx,0,4,
3,01234567,Ana Pérez,2026-02-03,Caracas,34,demo_pacientes_febrero.xlsx,0,2,ana@example.com
4,34567890,Pedro Díaz,2026-02-08,Barquisimeto,52,demo_pacientes_febrero.xlsx,0,3,pedro@example.com
5,45678901,José Martínez,2026-03-02,Maracay,38,demo_pacientes_marzo.xlsx,0,2,
6,01234567,Ana Pérez,2026-03-09,Caracas,34,demo_pacientes_marzo.xlsx,0,3,


## 10. Diagnóstico

Antes de exportar, revisamos:

- cuántas filas aportó cada archivo,
- cuántos valores vacíos quedaron por columna,
- una muestra con la trazabilidad completa.

Esto permite detectar un mapeo equivocado (por ejemplo, una columna que quedó 100 % vacía) sin abrir el Excel.


In [10]:
print("Filas por archivo de origen:")
display(CONSOLIDADO["archivo_origen"].value_counts().rename("filas").to_frame())

print("\nValores vacíos por columna:")
vacias = CONSOLIDADO.apply(lambda col: col.map(es_vacio)).sum()
display(vacias.rename("vacíos").to_frame())

print("\nMuestra con trazabilidad:")
columnas_muestra = ["archivo_origen", "hoja_origen", "fila_origen"]
columnas_muestra += [c for c in ENCABEZADO_REF[:3] if c not in columnas_muestra]
display(CONSOLIDADO[columnas_muestra].head(10))


Filas por archivo de origen:


,filas
archivo_origen,
demo_pacientes_enero.xlsx,3
demo_pacientes_febrero.xlsx,2
demo_pacientes_marzo.xlsx,2



Valores vacíos por columna:


,vacíos
CEDULA,0
NOMBRE,0
FECHA,0
SEDE,0
EDAD,0
archivo_origen,0
hoja_origen,0
fila_origen,0
CORREO,5



Muestra con trazabilidad:


,archivo_origen,hoja_origen,fila_origen,CEDULA,NOMBRE,FECHA
0,demo_pacientes_enero.xlsx,0,2,01234567,Ana Pérez,2026-01-05
1,demo_pacientes_enero.xlsx,0,3,12345678,Luis Gómez,2026-01-07
2,demo_pacientes_enero.xlsx,0,4,23456789,María Rodríguez,2026-01-12
3,demo_pacientes_febrero.xlsx,0,2,01234567,Ana Pérez,2026-02-03
4,demo_pacientes_febrero.xlsx,0,3,34567890,Pedro Díaz,2026-02-08
5,demo_pacientes_marzo.xlsx,0,2,45678901,José Martínez,2026-03-02
6,demo_pacientes_marzo.xlsx,0,3,01234567,Ana Pérez,2026-03-09


## 11. Exportar

Guardamos el consolidado en `data/salida/` con fecha y hora en el nombre para no pisar versiones anteriores. Después lo releemos para verificar que el archivo escrito tiene la misma cantidad de filas y columnas.

> Los datos se guardan como texto. Eso preserva cédulas con ceros a la izquierda (`01234567`) y evita que Excel las convierta a notación científica. Si después necesitás fechas o números reales, se convierten en un paso aparte.


In [11]:
marca_tiempo = datetime.now().strftime("%Y%m%d_%H%M%S")
RUTA_SALIDA = DIR_SALIDA / f"consolidado_{marca_tiempo}.xlsx"

CONSOLIDADO.to_excel(RUTA_SALIDA, index=False, sheet_name="CONSOLIDADO")

# Verificación: releer el archivo generado.
verificacion = pd.read_excel(RUTA_SALIDA, dtype=str, keep_default_na=False)
assert verificacion.shape == CONSOLIDADO.shape, (
    f"La relectura no coincide: {verificacion.shape} vs {CONSOLIDADO.shape}"
)

print(f"✅ Guardado en: {RUTA_SALIDA}")
print(f"   {verificacion.shape[0]} filas x {verificacion.shape[1]} columnas")
display(verificacion.head(5))


✅ Guardado en: /home/luis/Desktop/CORPOSALUD-TOOLS/data/salida/consolidado_20261026_180921.xlsx
   7 filas x 9 columnas


,CEDULA,NOMBRE,FECHA,SEDE,EDAD,archivo_origen,hoja_origen,fila_origen,CORREO
0,01234567,Ana Pérez,2026-01-05,Caracas,34,demo_pacientes_enero.xlsx,0,2,
1,12345678,Luis Gómez,2026-01-07,Valencia,41,demo_pacientes_enero.xlsx,0,3,
2,23456789,María Rodríguez,2026-01-12,Maracaibo,29,demo_pacientes_enero.xlsx,0,4,
3,01234567,Ana Pérez,2026-02-03,Caracas,34,demo_pacientes_febrero.xlsx,0,2,ana@example.com
4,34567890,Pedro Díaz,2026-02-08,Barquisimeto,52,demo_pacientes_febrero.xlsx,0,3,pedro@example.com


## 12. Función reutilizable (puente hacia la GUI)

Esta función junta todo el flujo en una sola llamada. Es la que después vamos a mover a un módulo `.py` (por ejemplo `excel_merge_core.py`) para que la use la interfaz Tkinter de la fase 2.

Parámetros principales:

- `rutas`: lista de archivos Excel.
- `archivo_referencia`: cuál define el encabezado.
- `hoja_referencia`: hoja de la referencia (por defecto la primera).
- `hojas_por_archivo`: overrides de hoja para archivos puntuales.
- `ajustes`: el mapeo manual.
- `extras`: columnas extra aprobadas.
- `guardar_en`: si se indica, escribe el `.xlsx`.

> Esta celda usa las funciones definidas en los pasos anteriores, así que hay que ejecutarla después de ellas.


In [12]:
def combinar_excels(
    rutas: list[str | Path],
    archivo_referencia: str,
    hoja_referencia: str | int = 0,
    hojas_por_archivo: dict[str, str | int] | None = None,
    ajustes: dict[str, dict[str, str | None]] | None = None,
    usar_sugerencias: bool = True,
    extras: dict[str, list[str]] | None = None,
    incluir_todos_los_extras: bool = False,
    guardar_en: str | Path | None = None,
) -> pd.DataFrame:
    """Combina Excel usando la fila 1 del archivo de referencia como encabezado."""
    rutas = [Path(r).expanduser().resolve() for r in rutas]
    hojas_por_archivo = {archivo_referencia: hoja_referencia, **(hojas_por_archivo or {})}
    ajustes = ajustes or {}
    extras = extras or {}

    perfiles = {
        ruta.name: cargar_perfil(ruta, hojas_por_archivo.get(ruta.name, 0))
        for ruta in rutas
    }
    if archivo_referencia not in perfiles:
        raise FileNotFoundError(f"No se cargó {archivo_referencia!r}.")
    columnas_ref = perfiles[archivo_referencia]["columnas"]

    mapeo = construir_mapeo(perfiles, columnas_ref, ajustes, usar_sugerencias)
    extras_detectadas = {
        nombre: [c for c in perfil["columnas"] if c not in columnas_ref]
        for nombre, perfil in perfiles.items()
    }
    if incluir_todos_los_extras:
        extras = extras_detectadas

    problemas = validar_configuracion(perfiles, mapeo, extras, columnas_ref)
    if problemas:
        raise ValueError("Revisá el mapeo:\n- " + "\n- ".join(problemas))

    consolidado = construir_consolidado(perfiles, mapeo, extras, columnas_ref)

    if guardar_en is not None:
        destino = Path(guardar_en).expanduser().resolve()
        destino.parent.mkdir(parents=True, exist_ok=True)
        consolidado.to_excel(destino, index=False, sheet_name="CONSOLIDADO")
        print(f"Guardado en: {destino}")

    return consolidado


# Prueba rápida con los mismos archivos (sin volver a guardar).
prueba = combinar_excels(
    rutas=ARCHIVOS,
    archivo_referencia=ARCHIVO_REFERENCIA,
    hoja_referencia=HOJA_REFERENCIA,
    hojas_por_archivo=HOJAS_POR_ARCHIVO,
    ajustes=AJUSTES_MANUALES,
    usar_sugerencias=USAR_SUGERENCIAS,
    extras=EXTRAS_A_INCLUIR,
)
print("Filas combinadas por la función:", len(prueba))


Filas combinadas por la función: 7


## Siguiente paso

Con la maqueta validada, la fase 2 es:

1. Mover estas funciones a `excel_merge_core.py`.
2. Armar la GUI Tkinter con un desplegable para elegir el archivo de referencia.
3. Reutilizar el mapeo manual y el aviso de columnas extra en la interfaz.

> En esta máquina falta `tkinter` (`python3-tk`), así que la GUI se instala y prueba en la fase 2.
